# Framings

This notebook loads the article dataframe and uses an LLM to extract frames from the articles according to the hero/villain/victim framing.

## Section 0: Setup

### Imports

In [ ]:
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from pydantic import BaseModel, Field

from news_agent_utils.events import ENTITY_LISTS_BY_EVENT
from news_agent_utils.io import read_table, save_table
from news_agent_utils.llm import build_llm_cache_key, create_provider_client, load_provider_api_key, run_cached_openai_structured_call

pd.set_option("display.max_colwidth", 160)


### Directories

In [ ]:
BASE_DIR = Path(".")

LLM_CACHE_DIR = BASE_DIR / "cache" / "framings" / "llm"
DATA_DIR = BASE_DIR / "data"

for path in [LLM_CACHE_DIR, DATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Experiment directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"LLM cache directory: {LLM_CACHE_DIR}")


### API Keys

In [ ]:
OPENAI_API_KEY = load_provider_api_key("openai")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY before running this notebook.")

OPENAI_CLIENT = create_provider_client("openai", OPENAI_API_KEY, openai_client_cls=OpenAI)
print("Loaded OpenAI API key.")


### LLM Cache

In [ ]:
# Shared structured OpenAI cache/call helper is imported from news_agent_utils.llm.
LLM_CACHE_READ_ENABLED = True


## Section 1: Load Articles

In [ ]:
ARTICLE_TABLE_NAME = "df_articles" # set this to the articles dataframe
df_articles = read_table(DATA_DIR, ARTICLE_TABLE_NAME)
df_articles.columns


In [ ]:
df_articles.event.value_counts()

In [ ]:
df_articles.source.value_counts()

In [ ]:
EXPECTED_SLANTS = {"left", "center", "right"}
required_article_columns = {"event", "title", "body", "url", "slant"}
assert required_article_columns.issubset(df_articles.columns)
assert set(df_articles["slant"].dropna().unique()).issubset(EXPECTED_SLANTS)
assert df_articles["article_id"].is_unique
print(f"Loaded {len(df_articles)} article rows across {df_articles['event'].nunique() if not df_articles.empty else 0} events.")


## Section 2: Define Entity Lists

In [ ]:
# Shared entity lists are imported from news_agent_utils.events.
ENTITY_LISTS_BY_EVENT


In [ ]:
def normalize_entities(entities: List[str]) -> List[str]:
    """Normalize entity lists by stripping whitespace, removing empties, and de-duplicating while preserving order."""
    ordered_entities = []
    seen = set()
    for entity in entities:
        normalized = str(entity).strip()
        if not normalized or normalized in seen:
            continue
        seen.add(normalized)
        ordered_entities.append(normalized)
    return ordered_entities


def format_entity_list_inline(entities: List[str]) -> str:
    return repr(list(entities))


ENTITY_LISTS_BY_EVENT = {
    str(event): normalize_entities(entities)
    for event, entities in ENTITY_LISTS_BY_EVENT.items()
}

article_events = set(df_articles["event"].unique())
entity_events = set(ENTITY_LISTS_BY_EVENT.keys())
missing_entity_events = sorted(article_events - entity_events)
assert not missing_entity_events, f"Missing entity lists for events: {missing_entity_events}"
assert all(ENTITY_LISTS_BY_EVENT[event] for event in article_events), "Every article event must have a non-empty entity list."

pd.DataFrame(
    [
        {"event": event, "entity_count": len(entities), "entities": entities}
        for event, entities in sorted(ENTITY_LISTS_BY_EVENT.items())
        if event in article_events
    ]
)

## Section 3: Extract Framings

In [ ]:
FRAMING_MODEL = "gpt-5.4-mini" # can adjust this based on available models 
FRAMING_TEMPERATURE = None 

print(f"Using framing model: {FRAMING_MODEL}")
df_articles = df_articles.copy()
df_articles["model"] = FRAMING_MODEL

### Defining Hero-Villain-Victim framing and extraction functions 

In [ ]:
class HeroVillainVictimSchema(BaseModel):
    hero: str = Field(description="The entity portrayed as a hero in the text. If no hero, return 'None'.")
    villain: str = Field(description="The entity portrayed as a villain in the text. If no villain, return 'None'.")
    victim: str = Field(description="The entity portrayed as a victim in the text. If no victim, return 'None'.")


def normalize_hvv_output(result: Dict[str, str], entity_list: List[str]) -> Dict[str, str]:
    allowed = set(entity_list) | {"None"}
    normalized = {}
    for role in ["hero", "villain", "victim"]:
        value = str(result.get(role, "None")).strip()
        normalized[role] = value if value in allowed else "None"
    return normalized

def invoke_structured_annotation(
    *,
    text: str,
    prompt: str,
    response_schema: Any,
    schema_name: str,
) -> tuple[Dict[str, Any], bool]:
    return run_cached_openai_structured_call(
        prompt=prompt + "\n\nText:\n" + text,
        model_name=FRAMING_MODEL,
        temperature=FRAMING_TEMPERATURE,
        schema_name=schema_name,
        response_schema=response_schema,
        llm_cache_dir=LLM_CACHE_DIR,
        openai_client=OPENAI_CLIENT,
        cache_read_enabled=LLM_CACHE_READ_ENABLED,
    )

def annotate_hvv(text: str, entity_list: List[str], prompt: str) -> Dict[str, Any]:
    cache_row, cache_hit = invoke_structured_annotation(
        text=text,
        prompt=prompt,
        response_schema=HeroVillainVictimSchema,
        schema_name="hero_villain_victim",
    )
    return {
        "cache_row": cache_row,
        "cache_hit": cache_hit,
        "framing_output": normalize_hvv_output(cache_row["response"], entity_list),
    }


In [ ]:
# validations for caching mechanism
assert (
    build_llm_cache_key(
        prompt="same prompt",
        model_name="demo-model",
        config_payload={"temperature": 0.7},
        output_mode="text",
    )
    == build_llm_cache_key(
        prompt="same prompt",
        model_name="demo-model",
        config_payload={"temperature": 0.7},
        output_mode="text",
    )
)
print("Article framing validation checks passed, before running framings.")

### Generate framings for all articles and save to a new dataframe

In [ ]:
FRAMINGS_TABLE_NAME = "df_framings"

def generate_framings_dataframe(
    df_articles: pd.DataFrame,
    entity_lists_by_event: Dict[str, List[str]],
) -> pd.DataFrame:
    extractor_specs = [
        {
            "extractor_name": "hero_villain_victim",
            "runner": annotate_hvv,
            "prompt_builder": lambda entities: (
                "You are a text annotation assistant. Identify the hero, villain, and victim in the text. "
                "Only choose from the provided entity list. If a role is absent, return 'None' for that role.\n\n"
                f"Entity list: {entities}"
            ),
        },
    ]

    rows = []
    cache_hit_count = 0
    provider_call_count = 0
    article_records = df_articles.to_dict(orient="records")
    total_jobs = len(article_records) * len(extractor_specs)

    with tqdm(total=total_jobs, desc="Framing articles") as progress:
        for idx, article_row in enumerate(article_records):
            if idx % 10 == 0 and idx > 0:
                print(f"Processed {idx} article(s)...")
            event = article_row["event"]
            entity_list = entity_lists_by_event[event]
            entity_list_prompt = format_entity_list_inline(entity_list)
            title = article_row.get("title", "") or ""
            body = article_row.get("body", "") or ""
            text = f"{title}\n\n{body}".strip()
       

            for extractor_spec in extractor_specs:
                prompt = extractor_spec["prompt_builder"](entity_list_prompt)
                full_prompt = prompt + "\n\nText:\n" + text
                result = extractor_spec["runner"](
                    text=text,
                    entity_list=entity_list,
                    prompt=prompt,
                )
                cache_row = result["cache_row"]
                if result["cache_hit"]:
                    cache_hit_count += 1
                else:
                    provider_call_count += 1
                rows.append(
                    {
                        "article_id": article_row["article_id"],
                        "event": article_row["event"],
                        "slant": article_row["slant"],
                        "title": article_row["title"],
                        "url": article_row["url"],
                        "extractor_name": extractor_spec["extractor_name"],
                        "entity_list": entity_list,
                        "prompt": full_prompt,
                        "model": cache_row["model"],
                        "temperature": cache_row["config"]["temperature"],
                        "llm_cache_key": cache_row["cache_key"],
                        "generated_at": cache_row["generated_at"],
                        "framing_output": result["framing_output"],
                    }
                )
                progress.update(1)

    print(
        "Framing cache summary: "
        f"{cache_hit_count} cache hit(s), "
        f"{provider_call_count} provider call(s)."
    )
    return pd.DataFrame(rows)

df_framings = generate_framings_dataframe(df_articles, ENTITY_LISTS_BY_EVENT)
save_table(df_framings, DATA_DIR, FRAMINGS_TABLE_NAME)
df_framings.columns

In [ ]:
expected_extractors = {"hero_villain_victim"}
assert set(df_framings["extractor_name"].unique()) == expected_extractors

framing_counts = (
    df_framings.groupby(["article_id", "extractor_name"])
    .size()
    .reset_index(name="row_count")
)
assert (framing_counts["row_count"] == 1).all()

print("Article framing validation checks passed.")